In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# Semantic Fashion Search Engine — Demo\n", "ARTIFICIA Assessment | Parth Shah"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "from transformers import CLIPModel, CLIPProcessor\n",
    "import faiss\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "from PIL import Image\n",
    "import matplotlib.pyplot as plt\n",
    "print('PyTorch:', torch.__version__)\n",
    "print('All imports working')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "MODEL_NAME = 'openai/clip-vit-base-patch32'\n",
    "model = CLIPModel.from_pretrained(MODEL_NAME)\n",
    "processor = CLIPProcessor.from_pretrained(MODEL_NAME)\n",
    "model.eval()\n",
    "catalog_df = pd.read_csv('catalog/metadata.csv')\n",
    "index = faiss.read_index('embeddings/faiss_index.bin')\n",
    "print(f'Catalog: {len(catalog_df)} items | FAISS: {index.ntotal} vectors')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def search_text(query, top_k=5):\n",
    "    inputs = processor(text=[query], return_tensors='pt', padding=True)\n",
    "    with torch.no_grad():\n",
    "        emb = model.get_text_features(**inputs)\n",
    "    emb = emb / emb.norm(dim=-1, keepdim=True)\n",
    "    vec = emb.squeeze().numpy().astype('float32').reshape(1, -1)\n",
    "    scores, indices = index.search(vec, top_k)\n",
    "    results = []\n",
    "    for idx, score in zip(indices[0], scores[0]):\n",
    "        row = catalog_df.iloc[idx]\n",
    "        results.append({'name': row['name'], 'category': row['category'],\n",
    "                        'score': round(float(score), 4),\n",
    "                        'image': f\"catalog/images/{row['image_file']}\"})\n",
    "    return results\n",
    "\n",
    "def show_results(results, title):\n",
    "    fig, axes = plt.subplots(1, len(results), figsize=(20, 5))\n",
    "    fig.suptitle(title, fontsize=14, fontweight='bold')\n",
    "    for res, ax in zip(results, axes):\n",
    "        ax.imshow(Image.open(res['image']))\n",
    "        ax.set_title(f\"{res['name']}\\n{res['category']}\\n{res['score']}\", fontsize=9)\n",
    "        ax.axis('off')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "print('Functions ready')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "results = search_text('casual blue jeans', top_k=5)\n",
    "show_results(results, 'Search: casual blue jeans')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "results = search_text('floral summer dress', top_k=5)\n",
    "show_results(results, 'Search: floral summer dress')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def search_image(image_path, top_k=5):\n",
    "    image = Image.open(image_path).convert('RGB')\n",
    "    inputs = processor(images=image, return_tensors='pt')\n",
    "    with torch.no_grad():\n",
    "        emb = model.get_image_features(**inputs)\n",
    "    emb = emb / emb.norm(dim=-1, keepdim=True)\n",
    "    vec = emb.squeeze().numpy().astype('float32').reshape(1, -1)\n",
    "    scores, indices = index.search(vec, top_k)\n",
    "    results = []\n",
    "    for idx, score in zip(indices[0], scores[0]):\n",
    "        row = catalog_df.iloc[idx]\n",
    "        results.append({'name': row['name'], 'category': row['category'],\n",
    "                        'score': round(float(score), 4),\n",
    "                        'image': f\"catalog/images/{row['image_file']}\"})\n",
    "    return results\n",
    "\n",
    "query_img = f\"catalog/images/{catalog_df.iloc[0]['image_file']}\"\n",
    "results = search_image(query_img, top_k=5)\n",
    "show_results(results, 'Image Search Results')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python", "version": "3.9.13"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}